# Quantile Forecasting (padronizado)


*By Miguel Ferreira*

**Este notebook, assim com todos os outros de cada ferramenta do envelope de risco, segue o mesmo protocolo:**
1. Importação do dataset e bibliotecas
2. Execução do ```setup()``` e alinhamento temporal
3. Construção da ferramenta de risco
4. Sanity checks mínimos
5. DataFrame final (features de risco) 
6. Padronização
7. Salvamento

Esta padronização é uma peça fundamental para um projeto **clean code.** Tanto que esta introdução estará presente em todos os notebooks de todas as ferramentas do envelope de risco.

---
A ferramenta deste notebook é o Quantile Forecasting.
> O Quantile Forecasting é uma ferramenta de risco **preditivo**

Isto a torna um pouco diferente de todas as outras, que são de *output pontual* (entregam uma análise para agora e não para o futuro).


## 1) Importação do dataset e bibliotecas


In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.append(str(Path("../../../src").resolve()))
from setup import setup

CSV_PATH = "../../../data/processed/financial_tools_datset.csv"
TARGET_COL = "Price"
HORIZON = 1
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15


## 2) Execução do `setup()` e alinhamento temporal


In [2]:
raw_df = pd.read_csv(CSV_PATH)
raw_df["Date"] = pd.to_datetime(raw_df["Date"], format="%m/%d/%Y")
raw_df = raw_df.sort_values("Date").set_index("Date")

splits = setup(
    csv_path=CSV_PATH,
    target_col=TARGET_COL,
    horizon=HORIZON,
    train_ratio=TRAIN_RATIO,
    val_ratio=VAL_RATIO,
    save_artifacts=False,
)

processed_df = raw_df.reset_index().copy()
if "Change %" in processed_df.columns:
    processed_df["Change %"] = (
        processed_df["Change %"].str.replace("%", "", regex=False).astype(float) / 100
    )

processed_df = processed_df.sort_values("Date")
processed_df["target_return"] = (
    processed_df[TARGET_COL].pct_change(HORIZON).shift(-HORIZON)
)
processed_df = processed_df.dropna().set_index("Date")

n_rows = len(processed_df)
train_end = int(n_rows * TRAIN_RATIO)
val_end = int(n_rows * (TRAIN_RATIO + VAL_RATIO))
val_index = processed_df.iloc[train_end:val_end].index

print("Validation length:", len(val_index))


Validation length: 205


## 3) Construção da ferramenta de risco (quantile forecasting)


In [3]:
from sklearn.ensemble import GradientBoostingRegressor

X_train = splits["X_train"]
X_val = splits["X_val"]
y_train = splits["y_train"]

quantiles = [0.05, 0.25, 0.50, 0.75, 0.95]
preds = {}

for q in quantiles:
    model = GradientBoostingRegressor(
        loss="quantile",
        alpha=q,
        n_estimators=300,
        learning_rate=0.05,
        max_depth=3,
        random_state=42,
    )
    model.fit(X_train, y_train)
    preds[q] = model.predict(X_val)

qf_dataset = pd.DataFrame(index=val_index)
qf_dataset["quantile_lower"] = preds[0.05]
qf_dataset["q25"] = preds[0.25]
qf_dataset["q50"] = preds[0.50]
qf_dataset["q75"] = preds[0.75]
qf_dataset["quantile_upper"] = preds[0.95]

qf_dataset["quantile_width"] = qf_dataset["quantile_upper"] - qf_dataset["quantile_lower"]
qf_dataset["quantile_skew"] = (
    (qf_dataset["q75"] - qf_dataset["q50"]) - (qf_dataset["q50"] - qf_dataset["q25"])
)

safe_width = qf_dataset["quantile_width"].replace(0, np.nan)
qf_dataset["asymmetry"] = (qf_dataset["q50"] - qf_dataset["quantile_lower"]) / safe_width
qf_dataset["asymmetry"] = qf_dataset["asymmetry"].replace([np.inf, -np.inf], np.nan).fillna(0.5)

qf_dataset = qf_dataset.sort_index()


## 4) Sanity checks mínimos


In [4]:
print("NaN críticos:")
print(qf_dataset[["quantile_lower", "q50", "quantile_upper", "quantile_width"]].isna().sum())

print("\nDistribuição plausível:")
print(qf_dataset.describe().T[["mean", "std", "min", "max"]])

print("\nAlinhamento temporal:")
print("Index monotonic increasing:", qf_dataset.index.is_monotonic_increasing)
print("Index has duplicates:", qf_dataset.index.has_duplicates)

print("Dataset lines == val_index lines:", len(qf_dataset) == len(val_index))
print("Dataset index == val_index:", qf_dataset.index.equals(val_index))


NaN críticos:
quantile_lower    0
q50               0
quantile_upper    0
quantile_width    0
dtype: int64

Distribuição plausível:
                    mean       std       min       max
quantile_lower -0.005247  0.001926 -0.008580 -0.001923
q25            -0.001333  0.002735 -0.009549  0.004326
q50             0.000290  0.002946 -0.008676  0.006839
q75             0.001668  0.002787 -0.006557  0.008716
quantile_upper  0.007153  0.001242  0.004511  0.009785
quantile_width  0.012400  0.001230  0.010031  0.017308
quantile_skew  -0.000244  0.001432 -0.004339  0.003468
asymmetry       0.450052  0.181334 -0.007338  0.831280

Alinhamento temporal:
Index monotonic increasing: True
Index has duplicates: False
Dataset lines == val_index lines: True
Dataset index == val_index: True


## 5) DataFrame final (features de risco) 


In [5]:
qf_dataset = qf_dataset[[
    "quantile_lower",
    "q25",
    "q50",
    "q75",
    "quantile_upper",
    "quantile_width",
    "quantile_skew",
    "asymmetry",
]]

qf_dataset.tail(), qf_dataset.shape


(            quantile_lower       q25       q50       q75  quantile_upper  \
 Date                                                                       
 2024-07-17       -0.007441 -0.003989 -0.002933 -0.001696        0.006761   
 2024-07-18       -0.006531 -0.001664  0.000717  0.002348        0.006761   
 2024-07-19       -0.006531 -0.003305 -0.002746 -0.001162        0.006862   
 2024-07-22       -0.006531 -0.004070 -0.004071 -0.003558        0.006862   
 2024-07-23       -0.006012 -0.001638 -0.000500  0.001136        0.008003   
 
             quantile_width  quantile_skew  asymmetry  
 Date                                                  
 2024-07-17        0.014202       0.000181   0.317459  
 2024-07-18        0.013292      -0.000749   0.545301  
 2024-07-19        0.013394       0.001024   0.282659  
 2024-07-22        0.013394       0.000513   0.183737  
 2024-07-23        0.014015       0.000498   0.393292  ,
 (205, 8))

## 6) Padronização via função única para indexação

In [6]:
def standardize_dataset(df):
    df = df.sort_index().bfill()

    df.index = pd.to_datetime(df.index)
    df.index = df.index.astype("datetime64[us]")
    df.index.name = "timestamp"

    df = df.astype(np.float32)

    assert isinstance(df.index, pd.DatetimeIndex)
    assert df.index.dtype == "datetime64[us]"
    assert df.index.name == "timestamp"
    assert df.index.is_monotonic_increasing
    assert not df.index.has_duplicates

    return df

In [7]:
qf_dataset = standardize_dataset(qf_dataset)

In [8]:
qf_dataset

,quantile_lower,q25,q50,q75,quantile_upper,quantile_width,quantile_skew,asymmetry
timestamp,,,,,,,,
2023-10-11,-0.007350,-0.009410,-0.006859,-0.005027,0.006894,0.014244,-0.000719,0.034526
2023-10-12,-0.002481,-0.001492,-0.001558,0.001844,0.008259,0.010740,0.003468,0.085892
2023-10-13,-0.002388,0.001441,0.002554,0.004484,0.008464,0.010852,0.000816,0.455458
2023-10-16,-0.004212,-0.002549,-0.001036,0.001298,0.007754,0.011966,0.000820,0.265448
2023-10-17,-0.006360,-0.003552,-0.003048,-0.000126,0.007754,0.014114,0.002417,0.234650
...,...,...,...,...,...,...,...,...
2024-07-17,-0.007441,-0.003989,-0.002933,-0.001696,0.006761,0.014202,0.000181,0.317459
2024-07-18,-0.006531,-0.001664,0.000717,0.002348,0.006761,0.013292,-0.000749,0.545301
2024-07-19,-0.006531,-0.003305,-0.002746,-0.001162,0.006862,0.013394,0.001024,0.282659


## 7) Salvamento (opcional)


In [9]:
OUTPUT_PATH = "../../../data/processed/qf/qf_features.parquet"
qf_dataset.to_parquet(OUTPUT_PATH, index=True)
OUTPUT_PATH


'../../../data/processed/qf/qf_features.parquet'